# 11 - Validate Hybrid Ternary Package

Valida que o pacote exportado contém 78 MLPs empacotados, reconstrói amostras via manifest e confirma que o adapter `l78` continua avaliando em `0.92`. O objetivo é validar pacote e metadados; não é runtime rápido.

In [ ]:
# Auto-bootstrap para Colab/local. Rode esta célula primeiro.
from pathlib import Path
import os, sys, json, gc, math

IN_COLAB = 'google.colab' in sys.modules
PROJECT_NAME = 'gemma4_theory'
COLAB_DRIVE_ROOT = '/content/drive/MyDrive/RPT'
LOCAL_ROOT = str(Path.cwd().resolve())

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = Path(COLAB_DRIVE_ROOT) / PROJECT_NAME
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    !pip -q install transformers accelerate sentencepiece safetensors peft datasets
else:
    PROJECT_ROOT = Path(LOCAL_ROOT) / PROJECT_NAME if Path(LOCAL_ROOT).name != PROJECT_NAME else Path(LOCAL_ROOT)
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

DATA_DIR = PROJECT_ROOT / 'data'
RESULTS_DIR = PROJECT_ROOT / 'results'
CHECKPOINTS_DIR = PROJECT_ROOT / 'checkpoints'
DEPLOY_DIR = PROJECT_ROOT / 'deploy'
UTILS_DIR = PROJECT_ROOT / 'utils'
for p in [DATA_DIR, RESULTS_DIR, CHECKPOINTS_DIR, DEPLOY_DIR, UTILS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('RESULTS_DIR =', RESULTS_DIR)
print('CHECKPOINTS_DIR =', CHECKPOINTS_DIR)
print('DEPLOY_DIR =', DEPLOY_DIR)


In [ ]:
import re
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import AutoPeftModelForCausalLM

HF_TOKEN = os.environ.get('HF_TOKEN')
PRUNED_CHECKPOINT = CHECKPOINTS_DIR / 'student_pruned'
L78_ADAPTER = CHECKPOINTS_DIR / 'student_ternary_recovered_l78'
MANIFEST_PATH = DEPLOY_DIR / 'manifest.json'
BIN_PATH = DEPLOY_DIR / 'mlp_ternary_2bit.bin'
QUESTIONS_PATH = DATA_DIR / 'gemma4_100_questions.json'
MAX_NEW_TOKENS = 24
BATCH_SIZE = 1
PROMPT_TEMPLATE = 'Responda de forma curta e direta.\nPergunta: {question}\nResposta:'

print('PRUNED_CHECKPOINT', PRUNED_CHECKPOINT.exists(), PRUNED_CHECKPOINT)
print('L78_ADAPTER', L78_ADAPTER.exists(), L78_ADAPTER)
print('MANIFEST', MANIFEST_PATH.exists(), MANIFEST_PATH)
print('BIN', BIN_PATH.exists(), BIN_PATH, BIN_PATH.stat().st_size if BIN_PATH.exists() else None)


In [ ]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

manifest = load_json(MANIFEST_PATH)
questions = load_json(QUESTIONS_PATH)

assert manifest['summary']['num_tensors'] == 78, manifest['summary']
assert BIN_PATH.exists() and BIN_PATH.stat().st_size == manifest['summary']['packed_total_bytes']
print(json.dumps(manifest['summary'], ensure_ascii=False, indent=2))


In [ ]:
def unpack_2bit_slice(blob, n, shape, scale):
    raw = torch.tensor(list(blob), dtype=torch.uint8)
    codes = torch.empty(raw.numel() * 4, dtype=torch.uint8)
    codes[0::4] = raw & 0b11
    codes[1::4] = (raw >> 2) & 0b11
    codes[2::4] = (raw >> 4) & 0b11
    codes[3::4] = (raw >> 6) & 0b11
    codes = codes[:n]
    vals = torch.zeros(n, dtype=torch.float32)
    vals[codes == 1] = scale
    vals[codes == 2] = -scale
    return vals.view(shape)

# Verify first, middle, last tensors reconstruct structurally.
with open(BIN_PATH, 'rb') as f:
    selected = [manifest['tensors'][0], manifest['tensors'][len(manifest['tensors'])//2], manifest['tensors'][-1]]
    checks = []
    for item in selected:
        f.seek(item['offset'])
        blob = f.read(item['packed_bytes'])
        tensor = unpack_2bit_slice(blob, item['numel'], tuple(item['shape']), item['scale'])
        uniq = sorted(float(x) for x in torch.unique(tensor).tolist())
        checks.append({'name': item['name'], 'shape': item['shape'], 'unique_values_count': len(uniq), 'unique_values': uniq[:5]})
print(json.dumps(checks, ensure_ascii=False, indent=2))


In [ ]:
# Shared local evaluator.
def normalize_text(text):
    text = '' if text is None else str(text)
    text = text.strip().lower().replace('**', '').replace('`', '')
    text = text.translate(str.maketrans('₀₁₂₃₄₅₆₇₈₉', '0123456789'))
    text = re.sub(r'\s+', ' ', text)
    for a,b in [('á','a'),('à','a'),('ã','a'),('â','a'),('é','e'),('ê','e'),('í','i'),('ó','o'),('ô','o'),('õ','o'),('ú','u'),('ç','c')]:
        text = text.replace(a,b)
    return text

def first_sentence(text):
    text = (text or '').strip()
    parts = re.split(r'(?<=[\.!?])\s+', text)
    return parts[0] if parts and parts[0] else text

def answer_is_correct(prediction, accepted_answers):
    pred = normalize_text(first_sentence(prediction))
    accepted = [normalize_text(a) for a in accepted_answers]
    return pred in accepted or any(ans in pred for ans in accepted if ans)

def parse_answer(full_text):
    return full_text.split('Resposta:')[-1].strip()

def run_eval(model, tokenizer, questions):
    rows=[]
    device = next(model.parameters()).device
    for q in questions:
        prompt = PROMPT_TEMPLATE.format(question=q['question'])
        inputs = tokenizer([prompt], return_tensors='pt', padding=True, truncation=True).to(device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
        raw = tokenizer.batch_decode(out, skip_special_tokens=True)[0]
        ans = parse_answer(raw)
        ok = answer_is_correct(ans, q['accepted_answers'])
        rows.append({'id': q['id'], 'category': q['category'], 'question': q['question'], 'expected_answer': q['expected_answer'], 'model_answer': ans, 'is_correct': ok, 'raw_output': raw})
    total=len(rows); correct=sum(1 for r in rows if r['is_correct'])
    by_cat={}
    for cat in sorted(set(r['category'] for r in rows)):
        cr=[r for r in rows if r['category']==cat]
        by_cat[cat]={'total':len(cr),'correct':sum(1 for r in cr if r['is_correct']),'accuracy':sum(1 for r in cr if r['is_correct'])/len(cr)}
    return rows, {'total_questions':total,'correct':correct,'accuracy':correct/total,'by_category':by_cat}


In [ ]:
# Validate functional adapter checkpoint. This does not yet execute packed MLP kernels;
# it validates that the recovery artifact still reproduces the l78 functional result.
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(str(L78_ADAPTER), token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoPeftModelForCausalLM.from_pretrained(str(L78_ADAPTER), token=HF_TOKEN, device_map='auto')
model.eval()
rows, summary = run_eval(model, tokenizer, questions)
print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
out = {
    'package_manifest': str(MANIFEST_PATH),
    'package_bin': str(BIN_PATH),
    'adapter': str(L78_ADAPTER),
    'manifest_summary': manifest['summary'],
    'structural_unpack_checks': checks,
    'functional_eval_summary': summary,
    'rows': rows,
    'note': 'Functional eval uses the l78 PEFT adapter. Packed MLP structural validation verifies the exported 2-bit package; fast packed runtime is not implemented here.'
}
out_path = RESULTS_DIR / 'deployment_audit' / 'hybrid_package_validation.json'
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding='utf-8')
print('saved', out_path)
